# importacao e criação de uma base consolidada e limpa

In [130]:
import pandas as pd
import numpy as np
from datetime import datetime

In [131]:
INPUT_FILE = '/content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
OUTPUT_DIR = '/mnt/user-data/outputs'

In [132]:
  df_2022 = pd.read_excel(INPUT_FILE, sheet_name='PEDE2022')
  df_2023 = pd.read_excel(INPUT_FILE, sheet_name='PEDE2023')
  df_2024 = pd.read_excel(INPUT_FILE, sheet_name='PEDE2024')

In [133]:
## melhor todos os RAs numa unica base

todos_ras = pd.concat([
    df_2022[['RA']],
    df_2023[['RA']],
    df_2024[['RA']]
]).drop_duplicates()

df_base = pd.DataFrame({'RA': todos_ras['RA'].values})

In [134]:
def extrair_ano_nascimento(valor):
    """Extrai ano de nascimento de diferentes formatos"""
    if pd.isna(valor):
        return np.nan
    try:
        # Se já for um ano (número entre 1900 e 2024)
        if isinstance(valor, (int, float)) and 1900 <= valor <= 2024:
            return int(valor)
        # Se for data
        data = pd.to_datetime(valor, errors='coerce')
        if not pd.isna(data):
            return data.year
        return np.nan
    except:
        return np.nan

In [135]:
map_2022 = df_2022.set_index('RA').to_dict('index')
map_2023 = df_2023.set_index('RA').to_dict('index')
map_2024 = df_2024.set_index('RA').to_dict('index')

In [136]:
def obter_ano_nascimento(ra):
    ano = extrair_ano_nascimento(map_2024.get(ra, {}).get('Data de Nasc'))
    if pd.notna(ano):
        return ano

    ano = extrair_ano_nascimento(map_2023.get(ra, {}).get('Data de Nasc'))
    if pd.notna(ano):
        return ano

    ano = extrair_ano_nascimento(map_2022.get(ra, {}).get('Ano nasc'))
    if pd.notna(ano):
        return ano

    idade_2024 = map_2024.get(ra, {}).get('Idade')
    if pd.notna(idade_2024):
        try:
            return 2024 - int(idade_2024)
        except:
            pass

    idade_2023 = map_2023.get(ra, {}).get('Idade')
    if pd.notna(idade_2023):
        try:
            return 2023 - int(idade_2023)
        except:
            pass

    idade_2022 = map_2022.get(ra, {}).get('Idade 22')
    if pd.notna(idade_2022):
        try:
            return 2022 - int(idade_2022)
        except:
            pass

    return np.nan

df_base['ano_nascimento'] = df_base['RA'].apply(obter_ano_nascimento)

In [137]:
df_base['idade_atual'] = 2025 - df_base['ano_nascimento']

In [138]:
def padronizar_genero(ra):
    genero = (map_2024.get(ra, {}).get('Gênero') or
              map_2023.get(ra, {}).get('Gênero') or
              map_2022.get(ra, {}).get('Gênero'))
    if genero in ['M', 'm', 'Masculino']:
        return 'Masculino'
    elif genero in ['F', 'f', 'Feminino', 'Menina']:
        return 'Feminino'
    return genero

df_base['genero'] = df_base['RA'].apply(padronizar_genero)

In [139]:
df_base['ano_ingresso'] = df_base['RA'].apply(
    lambda ra: map_2024.get(ra, {}).get('Ano ingresso') or
               map_2023.get(ra, {}).get('Ano ingresso') or
               map_2022.get(ra, {}).get('Ano ingresso')
)

In [140]:
def calcular_faixa_etaria(idade):
    if pd.isna(idade):
        return 'Não informado'
    elif idade < 10:
        return '6-9 anos'
    elif idade < 13:
        return '10-12 anos'
    elif idade < 16:
        return '13-15 anos'
    elif idade < 18:
        return '16-17 anos'
    else:
        return '18+ anos'

df_base['faixa_etaria'] = df_base['idade_atual'].apply(calcular_faixa_etaria)

In [141]:
def classificar_veterano_ingressante(ra):
    """
    Ingressante: Só tem dados em 2024
    Veterano: Tem dados em anos anteriores
    """
    tem_2022 = ra in map_2022
    tem_2023 = ra in map_2023
    tem_2024 = ra in map_2024

    if tem_2024 and not tem_2023 and not tem_2022:
        return 'Ingressante 2024'
    elif tem_2023 and tem_2024 and not tem_2022:
        return 'Ingressante 2023'
    elif tem_2022:
        return 'Veterano'
    else:
        return 'Ingressante'

df_base['status'] = df_base['RA'].apply(classificar_veterano_ingressante)

In [142]:
def consolidar_instituicoes(ra):
    """
    Se estudou em mais de uma, coloca as 3 separadas por ' | '
    Senão mantém a repetição
    """
    inst_2022 = map_2022.get(ra, {}).get('Instituição de ensino')
    inst_2023 = map_2023.get(ra, {}).get('Instituição de ensino')
    inst_2024 = map_2024.get(ra, {}).get('Instituição de ensino')

    # Remover nulos e normalizar
    instituicoes = []
    for inst in [inst_2022, inst_2023, inst_2024]:
        if pd.notna(inst):
            inst_str = str(inst).strip()
            if inst_str and inst_str not in ['nan', 'None']:
                instituicoes.append(inst_str)

    if not instituicoes:
        return np.nan

    # Contar únicas
    instituicoes_unicas = list(dict.fromkeys(instituicoes))  # Preserva ordem, remove duplicatas

    if len(instituicoes_unicas) > 1:
        # Mais de uma instituição
        return ' | '.join(instituicoes_unicas[:3])  # Até 3
    else:
        # Mesma instituição nos 3 anos
        return instituicoes_unicas[0]

df_base['instituicao_ensino'] = df_base['RA'].apply(consolidar_instituicoes)

In [143]:
mapeamentos_por_ano = {
    2022: {
        'df': df_2022,
        'map': map_2022,
        'colunas': {
            'fase': 'Fase',
            'pedra': 'Pedra 22',
            'inde': 'INDE 22',
            'classificacao_geral': 'Cg',
            'autoavaliacao': 'IAA',
            'engajamento': 'IEG',
            'psicossocial': 'IPS',
            'aprendizagem': 'IDA',
            'matematica': 'Matem',
            'portugues': 'Portug',
            'ingles': 'Inglês',
            'adequacao_nivel': 'IAN',
            'fase_ideal': 'Fase ideal'
        }
    },
    2023: {
        'df': df_2023,
        'map': map_2023,
        'colunas': {
            'fase': 'Fase',
            'pedra': 'Pedra 2023',
            'inde': 'INDE 2023',
            'classificacao_geral': 'Cg',
            'autoavaliacao': 'IAA',
            'engajamento': 'IEG',
            'psicossocial': 'IPS',
            'aprendizagem': 'IDA',
            'matematica': 'Mat',
            'portugues': 'Por',
            'ingles': 'Ing',
            'adequacao_nivel': 'IAN',
            'fase_ideal': 'Fase Ideal'
        }
    },
    2024: {
        'df': df_2024,
        'map': map_2024,
        'colunas': {
            'fase': 'Fase',
            'pedra': 'Pedra 2024',
            'inde': 'INDE 2024',
            'classificacao_geral': 'Cg',
            'autoavaliacao': 'IAA',
            'engajamento': 'IEG',
            'psicossocial': 'IPS',
            'aprendizagem': 'IDA',
            'matematica': 'Mat',
            'portugues': 'Por',
            'ingles': 'Ing',
            'adequacao_nivel': 'IAN',
            'fase_ideal': 'Fase Ideal'
        }
    }
}

for ano, config in mapeamentos_por_ano.items():
    mapa = config['map']
    colunas = config['colunas']

    for nome_padrao, nome_original in colunas.items():
        nome_coluna = f"{nome_padrao}_{ano}"
        df_base[nome_coluna] = df_base['RA'].apply(
            lambda ra: mapa.get(ra, {}).get(nome_original)
        )

        if nome_padrao not in ['fase', 'pedra', 'fase_ideal']:
            df_base[nome_coluna] = pd.to_numeric(df_base[nome_coluna], errors='coerce')


In [144]:
def padronizar_fase(fase_valor):
    """
    Padroniza formato da fase extraindo apenas o número
    Ex: 'FASE 8' -> 8, '8E' -> 8, '7A' -> 7
    """
    if pd.isna(fase_valor):
        return np.nan

    fase_str = str(fase_valor).strip().upper()

    fase_str = fase_str.replace('FASE ', '').replace('FASE', '')

    import re
    numeros = re.findall(r'\d+', fase_str)

    if numeros:
        try:
            return int(numeros[0])
        except:
            return np.nan

    try:
        return int(float(fase_str))
    except:
        return np.nan

# Aplicar padronização nas colunas de fase
for ano in [2022, 2023, 2024]:
    col_fase = f'fase_{ano}'
    if col_fase in df_base.columns:
        df_base[col_fase] = df_base[col_fase].apply(padronizar_fase)

In [145]:
def calcular_media_ou_ultimo(row, nome_base):
    """
    Calcula média se tiver os 3 anos.
    Senão usa o último valor disponível (2024 > 2023 > 2022).
    """
    val_2022 = row.get(f'{nome_base}_2022')
    val_2023 = row.get(f'{nome_base}_2023')
    val_2024 = row.get(f'{nome_base}_2024')

    valores = []
    for v in [val_2022, val_2023, val_2024]:
        if pd.notna(v):
            try:
                valores.append(float(v))
            except:
                pass

    if len(valores) == 3:
        return np.mean(valores)
    elif len(valores) > 0:
        if pd.notna(val_2024):
            return float(val_2024)
        elif pd.notna(val_2023):
            return float(val_2023)
        elif pd.notna(val_2022):
            return float(val_2022)

    return np.nan

In [146]:
indicadores = [
    'inde',
    'classificacao_geral',
    'autoavaliacao',
    'engajamento',
    'psicossocial',
    'aprendizagem',
    'matematica',
    'portugues',
    'ingles',
    'adequacao_nivel'
]

for indicador in indicadores:
    df_base[f'{indicador}_media'] = df_base.apply(
        lambda row: calcular_media_ou_ultimo(row, indicador),
        axis=1
    )

In [147]:
ordem_colunas = [
    'RA',
    'ano_nascimento',
    'idade_atual',
    'genero',
    'faixa_etaria',
    'ano_ingresso',
    'status',  # Veterano ou Ingressante
    'instituicao_ensino',
    'fase_2022', 'fase_2023', 'fase_2024',
    'pedra_2022', 'pedra_2023', 'pedra_2024',
    'inde_2022', 'inde_2023', 'inde_2024',
    'inde_media',
    'classificacao_geral_2022', 'classificacao_geral_2023', 'classificacao_geral_2024',
    'classificacao_geral_media',
    'autoavaliacao_2022', 'autoavaliacao_2023', 'autoavaliacao_2024',
    'autoavaliacao_media',
    'engajamento_2022', 'engajamento_2023', 'engajamento_2024',
    'engajamento_media',
    'psicossocial_2022', 'psicossocial_2023', 'psicossocial_2024',
    'psicossocial_media',
    'aprendizagem_2022', 'aprendizagem_2023', 'aprendizagem_2024',
    'aprendizagem_media',
    'matematica_2022', 'matematica_2023', 'matematica_2024',
    'matematica_media',
    'portugues_2022', 'portugues_2023', 'portugues_2024',
    'portugues_media',
    'ingles_2022', 'ingles_2023', 'ingles_2024',
    'ingles_media',
    'adequacao_nivel_2022', 'adequacao_nivel_2023', 'adequacao_nivel_2024',
    'adequacao_nivel_media',
    'fase_ideal_2022', 'fase_ideal_2023', 'fase_ideal_2024',
]

In [148]:
colunas_existentes = [col for col in ordem_colunas if col in df_base.columns]


In [149]:
colunas_restantes = [col for col in df_base.columns if col not in colunas_existentes]

df_final = df_base[colunas_existentes + colunas_restantes]

In [152]:
import os

csv_file = f'base_consolidada_personalizada.csv'
df_final.to_csv(csv_file, index=False, encoding='utf-8-sig')